# RAG (Retrieval Augmented Generation) in DemoGPT

RAG (Retrieval Augmented Generation) lets you build AI systems that answer questions based on **your own documents**. It works by combining two powerful techniques:

1. **Retrieval** - Finding the most relevant pieces of information from a vector store (a database optimized for semantic search)
2. **Generation** - Feeding those retrieved pieces to an LLM so it can generate accurate, grounded answers

This means your AI can answer questions about private documents, company knowledge bases, research papers, or any custom data -- without needing to fine-tune a model. DemoGPT makes this entire pipeline simple with its `BaseRAG` class.

In [ ]:
# Install DemoGPT
%pip install demogpt -q

## Setting Up Your API Key

Create a `.env` file in your project root with your OpenAI API key:

```
OPENAI_API_KEY=sk-your-key-here
```

Then load it with `python-dotenv` (already included with DemoGPT):

In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env")  # Loads OPENAI_API_KEY from .env file

import os
print("API key loaded:", "OPENAI_API_KEY" in os.environ)

## How RAG Works

The RAG pipeline follows these steps:

```
Documents -> Chunking -> Embedding -> Vector Store -> Retrieval -> LLM Generation
```

1. **Documents**: Your raw data (PDFs, text files, CSVs, URLs, etc.)
2. **Chunking**: Documents are split into smaller, manageable pieces
3. **Embedding**: Each chunk is converted into a numerical vector that captures its semantic meaning
4. **Vector Store**: These embeddings are stored in a vector database (Chroma, FAISS, Pinecone)
5. **Retrieval**: When a query comes in, the most similar chunks are retrieved from the vector store
6. **LLM Generation**: The retrieved chunks are passed to an LLM as context, which generates a final answer

DemoGPT's `BaseRAG` class handles **all of this automatically**. You just add your documents, ask questions, and get answers.

## Basic RAG Example

Let's create a simple RAG system. We'll add some text content and then query it.

In [ ]:
from demogpt_agenthub.rag import BaseRAG
from demogpt_agenthub.llms import OpenAIChatModel

llm = OpenAIChatModel(model_name="gpt-4o-mini")

rag = BaseRAG(
    llm=llm,
    vectorstore="chroma",
    persistent_path="rag_demo",
    index_name="demo_index",
    reset_vectorstore=True,
    k=4,
    verbose=False
)

# Add text content
rag.add_texts("DemoGPT is an open-source framework created by Melih Unsal. It can generate Streamlit apps from natural language and provides tools for building AI agents.")
rag.add_texts("DemoGPT supports multiple vector stores including Chroma, Pinecone, and FAISS for document retrieval.")

# Query the RAG
response = rag.run("What is DemoGPT and who created it?")
print(response)

## Adding Files

Beyond plain text, `BaseRAG` can ingest entire files. Use `add_files()` to load documents in various formats:

- **PDF** (`.pdf`) - Research papers, reports, manuals
- **TXT** (`.txt`) - Plain text files, notes
- **CSV** (`.csv`) - Tabular data
- **JSON** (`.json`) - Structured data
- **URLs** - Web pages and online articles

In [ ]:
# Adding files (example with various formats)
# rag.add_files(["document.pdf", "notes.txt", "data.csv"])

# Adding web content
# rag.add_files(["https://example.com/article"])

## Embedding Models

Embeddings convert text into numerical vectors for similarity search. You can choose different embedding models via the `embedding_model_name` parameter:

| Model | Type | Cost |
|-------|------|------|
| `sentence-transformers/all-mpnet-base-v2` | Local (HuggingFace) | Free |
| `text-embedding-3-small` | OpenAI API | Paid (cheapest) |
| `text-embedding-3-large` | OpenAI API | Paid (best quality) |
| `text-embedding-ada-002` | OpenAI API | Paid (legacy) |

By default, DemoGPT uses `sentence-transformers/all-mpnet-base-v2`, which runs locally and is completely free. Switch to OpenAI embeddings if you need higher quality or want to avoid downloading a local model.

In [ ]:
# Using OpenAI embeddings
rag_openai = BaseRAG(
    llm=llm,
    vectorstore="chroma",
    persistent_path="rag_openai_embed",
    index_name="openai_index",
    reset_vectorstore=True,
    embedding_model_name="text-embedding-3-small"
)

## Vector Store Options

DemoGPT supports three vector store backends. Choose the one that fits your use case:

- **Chroma** (`"chroma"`) - Lightweight and local. Great for development and prototyping. No external services needed.
- **FAISS** (`"faiss"`) - Facebook AI Similarity Search. Highly efficient for large datasets. Runs locally with optimized indexing.
- **Pinecone** (`"pinecone"`) - Cloud-hosted, fully managed. Production-ready with built-in scaling. Requires a Pinecone API key.

In [ ]:
# Chroma (local, lightweight)
rag_chroma = BaseRAG(
    llm=llm,
    vectorstore="chroma",
    persistent_path="rag_chroma",
    index_name="chroma_index",
    reset_vectorstore=True
)

# FAISS (local, optimized for large datasets)
# rag_faiss = BaseRAG(
#     llm=llm,
#     vectorstore="faiss",
#     persistent_path="rag_faiss",
#     index_name="faiss_index",
#     reset_vectorstore=True
# )

# Pinecone (cloud-hosted, requires API key)
# import os
# os.environ["PINECONE_API_KEY"] = "your-pinecone-api-key"
# rag_pinecone = BaseRAG(
#     llm=llm,
#     vectorstore="pinecone",
#     index_name="pinecone_index",
#     reset_vectorstore=True
# )

## Configuring Retrieval

Fine-tune how documents are retrieved:

- **`k`** - The number of document chunks to retrieve for each query. Higher values provide more context but may include less relevant results.
- **`filter`** - Apply filters to the retrieval process, such as a `score_threshold` to only return chunks above a minimum similarity score.

In [ ]:
rag = BaseRAG(
    llm=llm,
    vectorstore="chroma",
    persistent_path="rag_filtered",
    index_name="filtered_index",
    reset_vectorstore=True,
    k=2,
    filter={"search_kwargs": {"score_threshold": 0.5}}
)

## RAG as an Agent Tool

One of the most powerful features of DemoGPT is using RAG as a tool inside an agent. A `ReactAgent` can combine RAG with other tools like `PythonTool` to retrieve information and then perform computations on it.

In [ ]:
from demogpt_agenthub.agents import ReactAgent
from demogpt_agenthub.tools import PythonTool

rag = BaseRAG(
    llm=llm,
    vectorstore="chroma",
    persistent_path="agent_rag",
    index_name="agent_index",
    reset_vectorstore=True
)
rag.add_texts("The company revenue in 2024 was $5.2 million with a growth rate of 35%.")

agent = ReactAgent(
    tools=[rag, PythonTool()],
    llm=llm,
    verbose=True
)
response = agent.run("What was the company revenue? Calculate the projected revenue for next year using the growth rate.")
print(response)

## Persistent Storage

The `persistent_path` parameter controls where your vector store data is saved on disk. This is important for building up a knowledge base over time:

- **`persistent_path`** - Directory where the vector store files are saved. Your embeddings persist between sessions.
- **`reset_vectorstore=True`** - Clears the existing vector store and starts fresh. Use this when you want a clean slate.
- **`reset_vectorstore=False`** - Preserves existing data and adds new documents on top. Use this to incrementally build your knowledge base.

This means you can ingest documents once, and then query them in future sessions without re-processing.

In [ ]:
# First run: create and populate the knowledge base
rag_persistent = BaseRAG(
    llm=llm,
    vectorstore="chroma",
    persistent_path="my_knowledge_base",
    index_name="kb_index",
    reset_vectorstore=True  # Start fresh
)
rag_persistent.add_texts("Python was created by Guido van Rossum and first released in 1991.")
rag_persistent.add_texts("Python 3.12 introduced improved error messages and performance optimizations.")

# Later run: load existing data and add more
# rag_persistent = BaseRAG(
#     llm=llm,
#     vectorstore="chroma",
#     persistent_path="my_knowledge_base",
#     index_name="kb_index",
#     reset_vectorstore=False  # Keep existing data
# )
# rag_persistent.add_texts("Python 3.13 added experimental free-threaded mode.")

response = rag_persistent.run("When was Python first released?")
print(response)

## Summary

Key takeaways from this notebook:

- **RAG** combines document retrieval with LLM generation to answer questions grounded in your own data
- **`BaseRAG`** handles the entire pipeline: chunking, embedding, storing, retrieving, and generating
- **Add content** with `add_texts()` for raw text or `add_files()` for PDFs, CSVs, TXT, JSON, and URLs
- **Embedding models** can be local (free, default) or OpenAI-hosted (paid, higher quality)
- **Vector stores**: Chroma (local dev), FAISS (large-scale local), Pinecone (cloud production)
- **Retrieval tuning** with `k` (number of results) and `filter` (score threshold)
- **Agent integration**: Use RAG as a tool inside `ReactAgent` alongside other tools like `PythonTool`
- **Persistent storage** lets you build a knowledge base incrementally across sessions